In [0]:
bronze = spark.read.table("workspace.tfl.bronze_journeys")

In [0]:
def clean_column_names(data):
    data = data.withColumnsRenamed({'start_station_number': 'start_station_id', 'end_station_number': 'end_station_id', 'number': 'rental_id','start_station': 'start_station_name', 'end_station': 'end_station_name','start_date':'start_timestamp','end_date': 'end_timestamp'})
    return data
bronze = clean_column_names(bronze)


In [0]:
import pyspark.sql.functions as sf

bronze = bronze.withColumn('start_timestamp', sf.expr("try_to_timestamp(start_timestamp, 'yyyy-MM-dd HH:mm')"))
bronze = bronze.withColumn('end_timestamp', sf.expr("try_to_timestamp(end_timestamp, 'yyyy-MM-dd HH:mm')"))

In [0]:
bronze = bronze.withColumn('total_duration_ms', bronze["total_duration_ms"].cast("long"))

In [0]:
bronze.count()

In [0]:
bronze = bronze.dropDuplicates()

In [0]:
bronze.count()

In [0]:
bronze.orderBy("total_duration").show()

In [0]:
from pyspark.sql.functions import col

bronze = (bronze.where(col("total_duration") != "0s"))
bronze.count()

Dropped 6 rows with 0s duration.

In [0]:
bronze = bronze.where(col("total_duration_ms") < 8.64e+7)


In [0]:
bronze.count()

Dropped all 841 rows with a hire duration of over a day.

In [0]:
bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.tfl.silver_journeys")